In [12]:
import pandas as pd
import numpy as np
import re
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

class SentimentAnalyzer:
    def __init__(self, model_name='neuralmind/bert-base-portuguese-cased'):
        """
        Inicializa o analisador de sentimento com o modelo BERT português
        """
        print("Carregando modelo BERT português...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()  # Modo de avaliação
        
        # Palavras de referência para sentimentos
        self.positive_words = [
            "excelente", "ótimo", "bom", "maravilhoso", "perfeito", 
            "recomendo", "qualidade", "satisfeito", "amei", "gostei"
        ]
        self.negative_words = [
            "péssimo", "ruim", "horrível", "terrível", "decepcionante",
            "problema", "defeito", "insatisfeito", "odiei", "detestei"
        ]
        
        print("Modelo carregado com sucesso!")
    
    def preprocess_text(self, text):
        """
        Pré-processa o texto do review
        """
        if pd.isna(text) or text == '':
            return ""
        
        # Converte para string e minúsculas
        text = str(text).lower()
        
        # Remove URLs
        text = re.sub(r'http\S+|www\S+', '', text)
        
        # Remove emails
        text = re.sub(r'\S+@\S+', '', text)
        
        # Remove caracteres especiais mas mantém acentos e pontuação básica
        text = re.sub(r'[^\w\sáàâãéêíóôõúçñ.,!?-]', '', text)
        
        # Remove números isolados
        text = re.sub(r'\b\d+\b', '', text)
        
        # Remove espaços múltiplos
        text = re.sub(r'\s+', ' ', text)
        
        # Remove espaços no início e fim
        text = text.strip()
        
        return text
    
    def get_embeddings(self, texts):
        """
        Obtém embeddings BERT para uma lista de textos
        """
        embeddings = []
        
        for text in texts:
            if text == "":
                # Para textos vazios, usa embedding zero
                embeddings.append(torch.zeros(768))
                continue
            
            # Tokeniza o texto
            inputs = self.tokenizer(
                text,
                return_tensors='pt',
                truncation=True,
                padding=True,
                max_length=512
            )
            
            # Obtém embeddings sem calcular gradientes
            with torch.no_grad():
                outputs = self.model(**inputs)
                # Usa o embedding do token [CLS] como representação do texto
                embedding = outputs.last_hidden_state[:, 0, :].squeeze()
                embeddings.append(embedding)
        
        return torch.stack(embeddings)
    
    def calculate_sentiment_score(self, text_embeddings):
        """
        Calcula o score de sentimento baseado na similaridade com palavras de referência
        """
        # Obtém embeddings das palavras de referência
        positive_embeddings = self.get_embeddings(self.positive_words)
        negative_embeddings = self.get_embeddings(self.negative_words)
        
        # Calcula embedding médio para cada sentimento
        positive_mean = torch.mean(positive_embeddings, dim=0)
        negative_mean = torch.mean(negative_embeddings, dim=0)
        
        sentiment_scores = []
        
        for embedding in text_embeddings:
            # Calcula similaridade cosseno
            pos_sim = torch.cosine_similarity(embedding.unsqueeze(0), positive_mean.unsqueeze(0))
            neg_sim = torch.cosine_similarity(embedding.unsqueeze(0), negative_mean.unsqueeze(0))
            
            # Score final: diferença normalizada entre similaridades
            score = (pos_sim - neg_sim).item()
            
            # Normaliza para o intervalo [-1, 1] usando tanh
            normalized_score = np.tanh(score * 2)  # Multiplicador para amplificar diferenças
            
            sentiment_scores.append(normalized_score)
        
        return sentiment_scores

def analyze_dataframe_sentiment(df, column_name='review_comment_message'):
    """
    Analisa o sentimento de uma coluna de dataframe
    
    Args:
        df: DataFrame com os reviews
        column_name: Nome da coluna com os textos dos reviews
    
    Returns:
        DataFrame com colunas adicionais de análise de sentimento
    """
    # Verifica se a coluna existe
    if column_name not in df.columns:
        raise ValueError(f"Coluna '{column_name}' não encontrada no DataFrame")
    
    # Cria uma cópia do dataframe
    result_df = df.copy()
    
    # Inicializa o analisador
    analyzer = SentimentAnalyzer()
    
    print(f"Processando {len(df)} reviews...")
    
    # Pré-processa os textos
    print("Pré-processando textos...")
    processed_texts = [analyzer.preprocess_text(text) for text in df[column_name]]
    
    # Obtém embeddings
    print("Extraindo embeddings BERT...")
    embeddings = analyzer.get_embeddings(processed_texts)
    
    # Calcula scores de sentimento
    print("Calculando scores de sentimento...")
    sentiment_scores = analyzer.calculate_sentiment_score(embeddings)
    
    # Adiciona resultados ao dataframe
    result_df['processed_text'] = processed_texts
    result_df['sentiment_score'] = sentiment_scores
    
    # Classifica sentimentos
    result_df['sentiment_label'] = result_df['sentiment_score'].apply(
        lambda x: 'Positivo' if x > 0.1 else ('Negativo' if x < -0.1 else 'Neutro')
    )
    
    print("Análise concluída!")
    
    return result_df

def display_sentiment_summary(df_with_sentiment):
    """
    Exibe um resumo da análise de sentimento
    """
    print("\n" + "="*50)
    print("RESUMO DA ANÁLISE DE SENTIMENTO")
    print("="*50)
    
    # Estatísticas básicas
    sentiment_counts = df_with_sentiment['sentiment_label'].value_counts()
    total_reviews = len(df_with_sentiment)
    
    print(f"\nTotal de reviews analisados: {total_reviews}")
    print(f"\nDistribuição de sentimentos:")
    for sentiment, count in sentiment_counts.items():
        percentage = (count / total_reviews) * 100
        print(f"  {sentiment}: {count} ({percentage:.1f}%)")
    
    # Estatísticas dos scores
    scores = df_with_sentiment['sentiment_score']
    print(f"\nEstatísticas dos scores de sentimento:")
    print(f"  Média: {scores.mean():.3f}")
    print(f"  Mediana: {scores.median():.3f}")
    print(f"  Desvio padrão: {scores.std():.3f}")
    print(f"  Mínimo: {scores.min():.3f}")
    print(f"  Máximo: {scores.max():.3f}")
    
    # Exemplos de reviews mais positivos e negativos
    print(f"\n" + "-"*30)
    print("EXEMPLOS DE REVIEWS")
    print("-"*30)
    
    # Review mais positivo
    most_positive = df_with_sentiment.loc[df_with_sentiment['sentiment_score'].idxmax()]
    print(f"\nMais positivo (score: {most_positive['sentiment_score']:.3f}):")
    print(f"'{most_positive['review_comment_message'][:200]}...'")
    
    # Review mais negativo
    most_negative = df_with_sentiment.loc[df_with_sentiment['sentiment_score'].idxmin()]
    print(f"\nMais negativo (score: {most_negative['sentiment_score']:.3f}):")
    print(f"'{most_negative['review_comment_message'][:200]}...'")

# Exemplo de uso
if __name__ == "__main__":
    # Exemplo com dados fictícios
    sample_data = {
        'review_comment_message': [
            "Este produto é excelente! Recomendo muito, qualidade incrível.",
            "Péssimo produto, chegou com defeito e o atendimento foi terrível.",
            "Produto ok, nada demais mas cumpre o que promete.",
            "Amei! Superou minhas expectativas, muito bom mesmo.",
            "Não gostei, qualidade ruim e não vale o preço.",
            "",  # Review vazio
            "Produto mediano, nem bom nem ruim.",
            "Maravilhoso! Já comprei novamente, recomendo para todos."
        ]
    }
    
    # Cria DataFrame de exemplo
    df = pd.DataFrame(sample_data)
    
    print("DataFrame original:")
    print(df)
    
    # Analisa sentimentos
    df_analyzed = analyze_dataframe_sentiment(df)
    
    # Exibe resultados
    print("\n" + "="*80)
    print("RESULTADOS DA ANÁLISE")
    print("="*80)
    
    # Mostra alguns resultados
    columns_to_show = ['review_comment_message', 'sentiment_score', 'sentiment_label']
    print(df_analyzed[columns_to_show].to_string(index=False))
    
    # Exibe resumo
    display_sentiment_summary(df_analyzed)
    
    print("\n" + "="*80)
    print("Para usar com seus dados:")
    print("df_analyzed = analyze_dataframe_sentiment(seu_dataframe, 'nome_da_coluna')")
    print("="*80)

DataFrame original:
                              review_comment_message
0  Este produto é excelente! Recomendo muito, qua...
1  Péssimo produto, chegou com defeito e o atendi...
2  Produto ok, nada demais mas cumpre o que promete.
3  Amei! Superou minhas expectativas, muito bom m...
4     Não gostei, qualidade ruim e não vale o preço.
5                                                   
6                 Produto mediano, nem bom nem ruim.
7  Maravilhoso! Já comprei novamente, recomendo p...
Carregando modelo BERT português...
Modelo carregado com sucesso!
Processando 8 reviews...
Pré-processando textos...
Extraindo embeddings BERT...
Calculando scores de sentimento...
Análise concluída!

RESULTADOS DA ANÁLISE
                                           review_comment_message  sentiment_score sentiment_label
   Este produto é excelente! Recomendo muito, qualidade incrível.         0.077045          Neutro
Péssimo produto, chegou com defeito e o atendimento foi terrível.        -0.062668

In [14]:
df_analyzed.head(10)

,review_comment_message,processed_text,sentiment_score,sentiment_label
0,"Este produto é excelente! Recomendo muito, qua...","este produto é excelente! recomendo muito, qua...",0.077045,Neutro
1,"Péssimo produto, chegou com defeito e o atendi...","péssimo produto, chegou com defeito e o atendi...",-0.062668,Neutro
2,"Produto ok, nada demais mas cumpre o que promete.","produto ok, nada demais mas cumpre o que promete.",0.077706,Neutro
3,"Amei! Superou minhas expectativas, muito bom m...","amei! superou minhas expectativas, muito bom m...",0.133386,Positivo
4,"Não gostei, qualidade ruim e não vale o preço.","não gostei, qualidade ruim e não vale o preço.",0.010160,Neutro
5,,,0.000000,Neutro
6,"Produto mediano, nem bom nem ruim.","produto mediano, nem bom nem ruim.",0.036868,Neutro
7,"Maravilhoso! Já comprei novamente, recomendo p...","maravilhoso! já comprei novamente, recomendo p...",0.120443,Positivo
